In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

hf_token = os.getenv("HF_TOKEN")


from langchain_groq import ChatGroq
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=api_key, model="Gemma-7b-It")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x79a965b55810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x79a965b56ce0>, model_name='Gemma-7b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('imagination.pdf')
docs = loader.load_and_split()
docs = docs[:10]
docs

[Document(metadata={'source': 'imagination.pdf', 'page': 0}, page_content='Vol.:(0123456789)\nSynthese (2021) 199:7203–7230\nhttps://doi.org/10.1007/s11229-021-03110-x\n1 3\nIMAGINATION AND\xa0ITS LIMITS\nUntying the\xa0knot: imagination, perception and\xa0their neural \nsubstrates\nDan\xa0Cavedon‑Taylor1 \nReceived: 3 July 2020 / Accepted: 4 March 2021 / Published online: 26 March 2021 \n© The Author(s) 2021\nAbstract\nHow tight is the conceptual connection between imagination and perception? A \nnumber of philosophers, from the early moderns to present-day predictive process-\ning theorists, tie the knot as tightly as they can, claiming that states of the imagina-\ntion, i.e. mental imagery, are a proper subset of perceptual experience. This paper \nlabels such a view ‘perceptualism’ about the imagination and supplies new argu-\nments against it. The arguments are based on high-level perceptual content and, \ndistinctly, cognitive penetration. The paper also defuses a recent, influen

In [9]:
from langchain import PromptTemplate

generic_template = """
Write a short and concise summary of the following text:
Text: 
{text}
"""

prompt = PromptTemplate(
    input_variables=['text'],
    template = generic_template
)

prompt

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\nWrite a short and concise summary of the following text:\nText: \n{text}\n')

In [12]:
from langchain.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm, chain_type='stuff', prompt= prompt, verbose=True)
output_summary = chain.run(docs)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Write a short and concise summary of the following text:
Text: 
Vol.:(0123456789)
Synthese (2021) 199:7203–7230
https://doi.org/10.1007/s11229-021-03110-x
1 3
IMAGINATION AND ITS LIMITS
Untying the knot: imagination, perception and their neural 
substrates
Dan Cavedon‑Taylor1 
Received: 3 July 2020 / Accepted: 4 March 2021 / Published online: 26 March 2021 
© The Author(s) 2021
Abstract
How tight is the conceptual connection between imagination and perception? A 
number of philosophers, from the early moderns to present-day predictive process-
ing theorists, tie the knot as tightly as they can, claiming that states of the imagina-
tion, i.e. mental imagery, are a proper subset of perceptual experience. This paper 
labels such a view ‘perceptualism’ about the imagination and supplies new argu-
ments against it. The arguments are based on high-level perceptual content and, 
distinct

In [13]:
print(output_summary)

**Summary:**

The article argues that the traditional view of perceptualism, which claims that mental imagery is simply a subset of perceptual experience, is untenable. 

**Main Arguments:**

**1. High-Level Content:**
- Mental imagery can represent high-level properties


## Map reduce to Summerize Large Documents

In [14]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('imagination.pdf')
docs = loader.load_and_split()
# docs = docs[:10]
docs

[Document(metadata={'source': 'imagination.pdf', 'page': 0}, page_content='Vol.:(0123456789)\nSynthese (2021) 199:7203–7230\nhttps://doi.org/10.1007/s11229-021-03110-x\n1 3\nIMAGINATION AND\xa0ITS LIMITS\nUntying the\xa0knot: imagination, perception and\xa0their neural \nsubstrates\nDan\xa0Cavedon‑Taylor1 \nReceived: 3 July 2020 / Accepted: 4 March 2021 / Published online: 26 March 2021 \n© The Author(s) 2021\nAbstract\nHow tight is the conceptual connection between imagination and perception? A \nnumber of philosophers, from the early moderns to present-day predictive process-\ning theorists, tie the knot as tightly as they can, claiming that states of the imagina-\ntion, i.e. mental imagery, are a proper subset of perceptual experience. This paper \nlabels such a view ‘perceptualism’ about the imagination and supplies new argu-\nments against it. The arguments are based on high-level perceptual content and, \ndistinctly, cognitive penetration. The paper also defuses a recent, influen

In [18]:
final_documents = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100).split_documents(docs)
final_documents

[Document(metadata={'source': 'imagination.pdf', 'page': 0}, page_content='Vol.:(0123456789)\nSynthese (2021) 199:7203–7230\nhttps://doi.org/10.1007/s11229-021-03110-x\n1 3\nIMAGINATION AND\xa0ITS LIMITS\nUntying the\xa0knot: imagination, perception and\xa0their neural \nsubstrates\nDan\xa0Cavedon‑Taylor1 \nReceived: 3 July 2020 / Accepted: 4 March 2021 / Published online: 26 March 2021 \n© The Author(s) 2021\nAbstract\nHow tight is the conceptual connection between imagination and perception? A \nnumber of philosophers, from the early moderns to present-day predictive process-\ning theorists, tie the knot as tightly as they can, claiming that states of the imagina-\ntion, i.e. mental imagery, are a proper subset of perceptual experience. This paper \nlabels such a view ‘perceptualism’ about the imagination and supplies new argu-\nments against it. The arguments are based on high-level perceptual content and, \ndistinctly, cognitive penetration. The paper also defuses a recent, influen

In [21]:
chunks_prompt = """
Please summarize the below text:
Text: `{text}'
Summary:
"""

map_prompt_template = PromptTemplate(input_variables=['text'], template=chunks_prompt)
map_prompt_template

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template="\nPlease summarize the below text:\nText: `{text}'\nSummary:\n")

In [22]:
final_prompt = """
Provide the final summary if the entire text with these important points.
Add a Motivation title, Start the precise summary with an introduction and provide the summary in number points for the whole text.
Text: {text}

"""

final_prompt_template = PromptTemplate(input_variables=['text'], template= final_prompt)
final_prompt_template

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='\nProvide the final summary if the entire text with these important points.\nAdd a Motivation title, Start the precise summary with an introduction and provide the summary in number points for the whole text.\nText: {text}\n\n')

In [23]:
summary_chain = load_summarize_chain(
    llm= llm,
    chain_type= 'map_reduce',
    map_prompt= map_prompt_template,
    combine_prompt= final_prompt_template,
    verbose= True
)

output = summary_chain.run(final_documents)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Please summarize the below text:
Text: `Vol.:(0123456789)
Synthese (2021) 199:7203–7230
https://doi.org/10.1007/s11229-021-03110-x
1 3
IMAGINATION AND ITS LIMITS
Untying the knot: imagination, perception and their neural 
substrates
Dan Cavedon‑Taylor1 
Received: 3 July 2020 / Accepted: 4 March 2021 / Published online: 26 March 2021 
© The Author(s) 2021
Abstract
How tight is the conceptual connection between imagination and perception? A 
number of philosophers, from the early moderns to present-day predictive process-
ing theorists, tie the knot as tightly as they can, claiming that states of the imagina-
tion, i.e. mental imagery, are a proper subset of perceptual experience. This paper 
labels such a view ‘perceptualism’ about the imagination and supplies new argu-
ments against it. The arguments are based on high-level perceptual content and, 
distinctly, cognitive penetr

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (7726 > 1024). Running this sequence through the model will result in indexing errors




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Provide the final summary if the entire text with these important points.
Add a Motivation title, Start the precise summary with an introduction and provide the summary in number points for the whole text.
Text: **Summary:**

The paper challenges the prevailing philosophical view, 'perceptualism,' which argues that imagination and perception are closely linked. The author proposes that this view is inadequate and presents new arguments against it.

**Main Points:**

* **Cognitive penetration:** The paper argues that high-level perceptual content can be used to explain imagination, suggesting that imagination is not merely a perceptual state.
* **Neuroimaging evidence:** The claim that visual perception and mental imagery share a neural substrate in the primary visual cortex is questioned, citing evidence from aphantasia patients who exhibit dissociation between imagery and percept

In [25]:
print(output)

## Motivation:

This text sheds light on the intricate relationship between perception and imagination, highlighting the key differences and similarities between these two cognitive processes.

## Summary:

**1. Perception is constrained by physical reality:**
- Perception is limited by the characteristics of the physical world.
- High-level concepts are not directly represented in perceptual experience.


**2. Imagination transcends physical limitations:**
- Imagination can access and represent high-level concepts independently of physical properties.


**3. Cognitive penetration in imagination:**
- Imagination allows for conscious assignment of meaning and high-level content to visual imagery.


**4. Blurring the boundaries between perception and imagination:**
- Both perception and imagination involve conscious control and influence over sensory experiences.


**5. Implications for understanding consciousness:**
- Studying perception and imagination illuminates the relationship betw